# ML Regime Classification 4: LSTM Multi

## Overview
This Jupyter Notebook implements binary classification of volatility regimes using LSTM (Long Short-Term Memory) neural networks for temporal sequence learning. It tests a rolling window of 60 months (adjustable) and an expanding window to capture different cycle lengths, using walk-forward validation to ensure no lookahead bias. The model informs tactical allocation: LOW_VOL → offensive (high beta), HIGH_VOL → defensive (low beta).

## Key Features
- **Target**: Binary regime (1=HIGH_VOL if target_rv_max > median, 0=LOW_VOL otherwise).
- **Model**: Sequential LSTM (2 layers: 64 units + 32 units, Dropout 0.3, Dense sigmoid output).
- **Validation**: Walk-forward rolling (60 months, adjustable) and expanding windows, annual refit.
- **Strategies Tested**: Rolling 60m (business cycle), expanding (accumulating knowledge).
- **Allocation Rule**: Regime 0 → Offensive (high beta, growth), Regime 1 → Defensive (low beta, value).
- **Features**: 35 from momentum, dispersion, beta, cross-sectional, interactions.

## Methodology
1. Load features (35) and binarized volatility targets.
2. Test rolling window (60 months, adjustable) and expanding window strategies.
3. Walk-forward backtesting: Rolling or expanding train windows, predict next regime.
4. Fit LSTM model, predict regimes.
5. Evaluate: Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix.
6. Compare strategies, select best performer.

## Dependencies
- Libraries: tensorflow, keras, pandas, numpy, sklearn, matplotlib, seaborn, json, pickle.
- Data: data/ml_data/features/features_monthly.parquet, data/ml_data/volatility_targets/volatility_targets_y.parquet.
- Environment: RunPod A40 GPU (or CPU), Python 3.x.

## Usage
1. Run cells to load data, set strategy (rolling or expanding), adjust window size if needed.
2. Review metrics for each strategy, confusion matrix, ROC curve.
3. Compare with Logistic L2, Random Forest, XGBoost.
4. Use best strategy predictions for backtesting tactical allocation.

## Output Files
- **data/ml_data/models/lstm/{strategy}/predictions.parquet**: Predictions for each strategy.
- **data/ml_data/models/lstm/{strategy}/metrics.json**: Performance metrics per strategy.
- **data/ml_data/models/lstm/best_strategy_summary.json**: Winner selection across strategies.
- **data/ml_data/models/lstm/strategy_comparison.png**: Accuracy vs strategy plot.

This notebook explores temporal dependencies in regime classification, potentially outperforming tree-based models on sequential patterns.

In [12]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 0 : RUNPOD CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

USE_MODAL = False  # ❌ DISABLE BEAM (use RunPod instead)

print(f"✅ MODE : RUNPOD A40 GPU")
print(f"   • Expected runtime : ~8 min (4 windows)")
print(f"   • Cost : $0.027 (8min × $0.20/h)")

✅ MODE : RUNPOD A40 GPU
   • Expected runtime : ~8 min (4 windows)
   • Cost : $0.027 (8min × $0.20/h)


In [13]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 : IMPORTS (ENABLE GPU)
# ════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import os
# ✅ REMOVE CPU FORCING (enable GPU)
# os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # ← COMMENT THIS LINE
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import gc
import tensorflow as tf

print("="*100)
print("📦 NOTEBOOK 08_4 : REGIME CLASSIFICATION - LSTM (RUNPOD A40)")
print("="*100)

# Verify GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"\n✅ GPU Detection :")
print(f"   • Devices found : {len(gpus)}")
for gpu in gpus:
    print(f"   • {gpu}")

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

print(f"\n✅ TensorFlow : {tf.__version__}")
print(f"   • GPU enabled : {len(gpus) > 0}")
print(f"   • Expected runtime : ~8 min (A40)")
print("="*100)

tf.random.set_seed(42)
np.random.seed(42)

📦 NOTEBOOK 08_4 : REGIME CLASSIFICATION - LSTM (RUNPOD A40)

✅ GPU Detection :
   • Devices found : 1
   • PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

✅ TensorFlow : 2.16.2
   • GPU enabled : True
   • Expected runtime : ~8 min (A40)


In [14]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 2 : DATA LOADING (FIXED - KEEP PRE-1969 DATA)
# ════════════════════════════════════════════════════════════════════════════

print("="*100)
print("📦 DATA LOADING")
print("="*100)

# ════════════════════════════════════════════════════════════════════════════
# 1. LOAD FEATURES (35 features from Notebook 07_2)
# ════════════════════════════════════════════════════════════════════════════

FEATURES_PATH = Path('data/ml_data/features/features_monthly.parquet')
df_features = pd.read_parquet(FEATURES_PATH)

# Extract only the 35 features (remove the 4 targets from 07_2)
feature_cols = df_features.columns[:35].tolist()
X_all = df_features[feature_cols].copy()

print(f"\n✅ Features loaded :")
print(f"   • Source         : {FEATURES_PATH}")
print(f"   • Shape          : {X_all.shape}")
print(f"   • Period         : {X_all.index.min().date()} → {X_all.index.max().date()}")
print(f"   • Total months   : {len(X_all)}")

# ════════════════════════════════════════════════════════════════════════════
# 2. LOAD VOLATILITY TARGETS (ALREADY BINARIZED from Notebook 07_1)
# ════════════════════════════════════════════════════════════════════════════

TARGETS_PATH = Path('data/ml_data/volatility_targets/volatility_targets_y.parquet')
df_targets = pd.read_parquet(TARGETS_PATH)

print(f"\n✅ Volatility targets loaded :")
print(f"   • Source         : {TARGETS_PATH}")
print(f"   • Shape          : {df_targets.shape}")
print(f"   • Period         : {df_targets.index.min().date()} → {df_targets.index.max().date()}")

# ════════════════════════════════════════════════════════════════════════════
# 3. ✅ FIX : USE LEFT JOIN TO KEEP ALL FEATURES (even without targets)
# ════════════════════════════════════════════════════════════════════════════
# 
# We need pre-1969 features for rolling windows, even without targets
# Targets will be NaN before 1969-06 (dropped in CELL 3)
# ════════════════════════════════════════════════════════════════════════════

TARGET_COLUMN = 'Y_EWMA_BINARY'

# Left join: keep ALL features (targets=NaN before 1969-06)
df = X_all.join(df_targets[[TARGET_COLUMN]], how='left')

print(f"\n✅ Data merged (LEFT JOIN to keep pre-1969 features) :")
print(f"   • Final shape    : {df.shape}")
print(f"   • Period         : {df.index.min().date()} → {df.index.max().date()}")
print(f"   • Features       : {X_all.shape[0]} months")
print(f"   • Targets        : {df[TARGET_COLUMN].notna().sum()} months")

X_final = df[feature_cols].copy()
y_regime = df[TARGET_COLUMN].copy()

print(f"\n📊 Target distribution (where available) :")
y_valid = y_regime.dropna()
print(f"   • Regime 0 (LOW VOL)  : {(y_valid == 0).sum()} ({(y_valid == 0).mean()*100:.1f}%)")
print(f"   • Regime 1 (HIGH VOL) : {(y_valid == 1).sum()} ({(y_valid == 1).mean()*100:.1f}%)")
print(f"   • Missing (pre-1969)  : {y_regime.isna().sum()} months")

print("\n" + "="*100)

📦 DATA LOADING

✅ Features loaded :
   • Source         : data/ml_data/features/features_monthly.parquet
   • Shape          : (678, 35)
   • Period         : 1968-07-31 → 2024-12-31
   • Total months   : 678

✅ Volatility targets loaded :
   • Source         : data/ml_data/volatility_targets/volatility_targets_y.parquet
   • Shape          : (649, 6)
   • Period         : 1969-06-30 → 2023-06-30

✅ Data merged (LEFT JOIN to keep pre-1969 features) :
   • Final shape    : (678, 36)
   • Period         : 1968-07-31 → 2024-12-31
   • Features       : 678 months
   • Targets        : 649 months

📊 Target distribution (where available) :
   • Regime 0 (LOW VOL)  : 409 (63.0%)
   • Regime 1 (HIGH VOL) : 240 (37.0%)
   • Missing (pre-1969)  : 29 months



In [15]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 3 : DATA CLEANING (CORRECTED - KEEP ALL DATA INCLUDING PRE-1969)
# ════════════════════════════════════════════════════════════════════════════

print("="*100)
print("🧹 DATA CLEANING")
print("="*100)

# ✅ DO NOT DROP : We need pre-1969 features for rolling windows
# Just separate X and y (y will have NaN before 1969-06)

X = X_final.copy()
y = y_regime.copy()

print(f"\n✅ Data prepared (NO DROPNA) :")
print(f"   • X shape     : {X.shape}")
print(f"   • X period    : {X.index.min().date()} → {X.index.max().date()}")
print(f"   • y non-null  : {y.notna().sum()} (from {y.notna().idxmax().date()})")
print(f"   • y null      : {y.isna().sum()} (pre-1969 warmup)")

print(f"\n📊 Target distribution (non-null only) :")
y_valid = y.dropna()
print(f"   • Regime 0 (LOW VOL)  : {(y_valid == 0).sum()} ({(y_valid == 0).mean()*100:.1f}%)")
print(f"   • Regime 1 (HIGH VOL) : {(y_valid == 1).sum()} ({(y_valid == 1).mean()*100:.1f}%)")

# Compute scale_pos_weight for XGBoost
n_neg = (y_valid == 0).sum()
n_pos = (y_valid == 1).sum()
scale_pos_weight = n_neg / n_pos

print(f"\n⚖️  CLASS BALANCING (XGBoost) :")
print(f"   • scale_pos_weight = {scale_pos_weight:.4f}")
print(f"   • Interpretation   : Assign {scale_pos_weight:.2f}× weight to HIGH_VOL samples")

print(f"\n⚠️  IMPORTANT :")
print(f"   • Pre-1969 data KEPT in X (for rolling windows)")
print(f"   • Formation dates start at 1969-06 (where y becomes non-null)")
print(f"   • Each rolling window can use 1964-06+ features")

print("\n" + "="*100)

🧹 DATA CLEANING

✅ Data prepared (NO DROPNA) :
   • X shape     : (678, 35)
   • X period    : 1968-07-31 → 2024-12-31
   • y non-null  : 649 (from 1969-06-30)
   • y null      : 29 (pre-1969 warmup)

📊 Target distribution (non-null only) :
   • Regime 0 (LOW VOL)  : 409 (63.0%)
   • Regime 1 (HIGH VOL) : 240 (37.0%)

⚖️  CLASS BALANCING (XGBoost) :
   • scale_pos_weight = 1.7042
   • Interpretation   : Assign 1.70× weight to HIGH_VOL samples

⚠️  IMPORTANT :
   • Pre-1969 data KEPT in X (for rolling windows)
   • Formation dates start at 1969-06 (where y becomes non-null)
   • Each rolling window can use 1964-06+ features



In [16]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 4 : WALK-FORWARD BACKTESTING SETUP (ALIGNED WITH 08_1/08_2/08_3)
# ════════════════════════════════════════════════════════════════════════════

print("="*100)
print("📅 WALK-FORWARD BACKTESTING SETUP (FAMA-FRENCH ALIGNED)")
print("="*100)

# ════════════════════════════════════════════════════════════════════════════
# CONFIGURATION : ANNUAL REBALANCING (ALIGNED WITH OTHER MODELS)
# ════════════════════════════════════════════════════════════════════════════

FIRST_TEST_YEAR = 1980
LAST_TEST_YEAR = 2023

# ════════════════════════════════════════════════════════════════════════════
# STRATEGY SELECTION (RUN ONE AT A TIME)
# ════════════════════════════════════════════════════════════════════════════
# 
# IMPORTANT: Execute notebook TWICE (once per strategy)
# 
# Strategy 1 : EXPANDING WINDOW
#   • Train period : [1969-06 : year-1] (growing)
#   • Captures long-term trends, maximum stability
#   • Same as Logistic/RF/XGBoost (baseline)
#
# Strategy 2 : ROLLING 10-YEAR WINDOW
#   • Train period : [year-120m : year-1] (fixed 120 months)
#   • Captures business cycles (7-11 years typical)
#   • Adapts to regime shifts
# ════════════════════════════════════════════════════════════════════════════

# ✅ SET THIS PARAMETER (change between runs)
STRATEGY = 'rolling_10y'  # Options: 'expanding' OR 'rolling_10y'

# ════════════════════════════════════════════════════════════════════════════
# VALIDATION
# ════════════════════════════════════════════════════════════════════════════

if STRATEGY not in ['expanding', 'rolling_10y']:
    raise ValueError(f"Invalid STRATEGY: {STRATEGY}. Must be 'expanding' or 'rolling_10y'")

print(f"\n📊 CONFIGURATION :")
print(f"   • Strategy        : {STRATEGY.upper()}")
print(f"   • Test period     : {FIRST_TEST_YEAR}-06 → {LAST_TEST_YEAR}-06")
print(f"   • Rebalancing     : Annual (June)")
print(f"   • Formations      : {LAST_TEST_YEAR - FIRST_TEST_YEAR + 1}")

# ════════════════════════════════════════════════════════════════════════════
# DATA AVAILABILITY CHECK
# ════════════════════════════════════════════════════════════════════════════

y_first_valid = y.notna().idxmax()  # First non-NaN target (1969-06-30)

print(f"\n📊 DATA AVAILABILITY :")
print(f"   • Features start  : {X.index.min().date()}")
print(f"   • Targets start   : {y_first_valid.date()}")
print(f"   • Data end        : {X.index.max().date()}")

# ════════════════════════════════════════════════════════════════════════════
# STRATEGY DETAILS
# ════════════════════════════════════════════════════════════════════════════

print(f"\n📅 STRATEGY DETAILS : {STRATEGY.upper()}")

if STRATEGY == 'expanding':
    print(f"   • Type            : Expanding window")
    print(f"   • Train start     : {y_first_valid.date()} (FIXED)")
    print(f"   • Train end       : year - 1 (GROWING)")
    print(f"   • First formation : {FIRST_TEST_YEAR}-06 (train size = 132 months)")
    print(f"   • Last formation  : {LAST_TEST_YEAR}-06 (train size = 648 months)")
    print(f"   • Rationale       : Maximum stability, captures long-term trends")

elif STRATEGY == 'rolling_10y':
    print(f"   • Type            : Rolling 10-year window")
    print(f"   • Train period    : 120 months (FIXED)")
    print(f"   • Train start     : year - 120 months (ROLLING)")
    print(f"   • Train end       : year - 1 (ROLLING)")
    print(f"   • First formation : {FIRST_TEST_YEAR}-06 (train = 1970-06 to 1979-06)")
    print(f"   • Last formation  : {LAST_TEST_YEAR}-06 (train = 2013-06 to 2022-06)")
    print(f"   • Rationale       : Business cycle capture (7-11 years typical)")

print(f"\n⚖️  FAMA-FRENCH COMPLIANCE :")
print(f"   • ✅ NO LOOKAHEAD : Train uses only data BEFORE test date")
print(f"   • ✅ WALK-FORWARD : Model re-fitted every year")
print(f"   • ✅ ALIGNMENT    : Same dates as Logistic/RF/XGBoost (08_1/08_2/08_3)")

print("\n" + "="*100)


📅 WALK-FORWARD BACKTESTING SETUP (FAMA-FRENCH ALIGNED)

📊 CONFIGURATION :
   • Strategy        : ROLLING_10Y
   • Test period     : 1980-06 → 2023-06
   • Rebalancing     : Annual (June)
   • Formations      : 44

📊 DATA AVAILABILITY :
   • Features start  : 1968-07-31
   • Targets start   : 1969-06-30
   • Data end        : 2024-12-31

📅 STRATEGY DETAILS : ROLLING_10Y
   • Type            : Rolling 10-year window
   • Train period    : 120 months (FIXED)
   • Train start     : year - 120 months (ROLLING)
   • Train end       : year - 1 (ROLLING)
   • First formation : 1980-06 (train = 1970-06 to 1979-06)
   • Last formation  : 2023-06 (train = 2013-06 to 2022-06)
   • Rationale       : Business cycle capture (7-11 years typical)

⚖️  FAMA-FRENCH COMPLIANCE :
   • ✅ NO LOOKAHEAD : Train uses only data BEFORE test date
   • ✅ WALK-FORWARD : Model re-fitted every year
   • ✅ ALIGNMENT    : Same dates as Logistic/RF/XGBoost (08_1/08_2/08_3)



In [17]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 4.5 : MEMORY & PERFORMANCE OPTIMIZATION (MACOS)
# ════════════════════════════════════════════════════════════════════════════

import gc
import tensorflow as tf

print("="*100)
print("🧹 MEMORY CLEANUP & OPTIMIZATION")
print("="*100)

# ────────────────────────────────────────────────────────────────────────────
# 1. CLEAR PYTHON GARBAGE
# ────────────────────────────────────────────────────────────────────────────
print("\n🗑️  Clearing Python garbage collector...")
gc.collect()
print(f"   ✅ Freed memory objects")

# ────────────────────────────────────────────────────────────────────────────
# 2. CLEAR TENSORFLOW SESSION
# ────────────────────────────────────────────────────────────────────────────
print("\n🧠 Clearing TensorFlow session...")
tf.keras.backend.clear_session()
print(f"   ✅ TensorFlow backend reset")

# ────────────────────────────────────────────────────────────────────────────
# 3. DISABLE TENSORFLOW MEMORY GROWTH (avoid fragmentation)
# ────────────────────────────────────────────────────────────────────────────
print("\n💾 Configuring TensorFlow memory...")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"   ✅ GPU memory growth enabled (avoid fragmentation)")
    except RuntimeError as e:
        print(f"   ⚠️  Could not set memory growth: {e}")
else:
    print(f"   ℹ️  No GPU detected (CPU mode)")

# ────────────────────────────────────────────────────────────────────────────
# 4. SET TENSORFLOW THREAD LIMITS (prevent CPU overload)
# ────────────────────────────────────────────────────────────────────────────
print("\n🔧 Setting TensorFlow thread limits...")
tf.config.threading.set_intra_op_parallelism_threads(2)  # Limit TF internal threads
tf.config.threading.set_inter_op_parallelism_threads(2)  # Limit TF inter-op threads
print(f"   ✅ TensorFlow threads limited (avoid thermal throttling)")

# ────────────────────────────────────────────────────────────────────────────
# 5. FORCE NUMPY TO SINGLE THREAD (avoid contention with joblib)
# ────────────────────────────────────────────────────────────────────────────
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
print("\n🔢 NumPy threading disabled...")
print(f"   ✅ Avoid CPU contention with joblib workers")

print(f"\n{'='*100}")
print("✅ OPTIMIZATION COMPLETE - READY FOR TRAINING")
print(f"{'='*100}\n")

🧹 MEMORY CLEANUP & OPTIMIZATION

🗑️  Clearing Python garbage collector...
   ✅ Freed memory objects

🧠 Clearing TensorFlow session...
   ✅ TensorFlow backend reset

💾 Configuring TensorFlow memory...
   ✅ GPU memory growth enabled (avoid fragmentation)

🔧 Setting TensorFlow thread limits...
   ✅ TensorFlow threads limited (avoid thermal throttling)

🔢 NumPy threading disabled...
   ✅ Avoid CPU contention with joblib workers

✅ OPTIMIZATION COMPLETE - READY FOR TRAINING



In [18]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 5 : LSTM TRAINING (ANNUAL WALK-FORWARD)
# ════════════════════════════════════════════════════════════════════════════

def train_single_formation_annual(args):
    """
    Worker function for annual walk-forward.
    Trains ONE LSTM model for ONE formation year.
    """
    test_year, strategy, X_values, X_index, y_values, y_index = args
    
    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping
    import tensorflow as tf
    import warnings
    import os
    
    warnings.filterwarnings('ignore')
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    tf.get_logger().setLevel('ERROR')
    
    # Reconstruct DataFrames
    X = pd.DataFrame(X_values, index=pd.to_datetime(X_index))
    y = pd.Series(y_values, index=pd.to_datetime(y_index))
    
    def build_lstm_model(n_features):
        model = Sequential([
            LSTM(32, input_shape=(1, n_features)),
            Dropout(0.2),
            Dense(1, activation='sigmoid')
        ])
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        return model
    
    try:
        # ────────────────────────────────────────────────────────────────────
        # 1. DEFINE FORMATION DATE
        # ────────────────────────────────────────────────────────────────────
        formation_date = pd.Timestamp(f'{test_year}-06-30')
        
        if formation_date not in X.index:
            return None
        
        # ────────────────────────────────────────────────────────────────────
        # 2. GET TEST SAMPLE
        # ────────────────────────────────────────────────────────────────────
        y_test_val = y.loc[formation_date]
        
        if pd.isna(y_test_val):
            return None
        
        # ────────────────────────────────────────────────────────────────────
        # 3. DEFINE TRAIN PERIOD (STRATEGY-DEPENDENT)
        # ────────────────────────────────────────────────────────────────────
        train_end = pd.Timestamp(f'{test_year - 1}-06-30')
        
        if strategy == 'expanding':
            # Expanding: use ALL data from start
            train_start = X.index.min()  # 1964-06-30
        
        elif strategy == 'rolling_10y':
            # Rolling 10Y: use last 120 months
            train_start = train_end - pd.DateOffset(months=119)
        
        else:
            return None
        
        # ────────────────────────────────────────────────────────────────────
        # 4. EXTRACT TRAIN DATA
        # ────────────────────────────────────────────────────────────────────
        train_mask = (X.index >= train_start) & (X.index <= train_end)
        
        X_train_raw = X.loc[train_mask].values
        y_train_raw = y.loc[train_mask].values
        
        # Remove NaN targets (pre-1969 warmup)
        valid_mask = ~np.isnan(y_train_raw)
        n_valid = valid_mask.sum()
        
        if n_valid < 50:  # Minimum 50 samples
            return None
        
        X_train = X_train_raw[valid_mask]
        y_train = y_train_raw[valid_mask]
        
        # Check class balance
        if len(np.unique(y_train)) < 2:
            return None
        
        # ────────────────────────────────────────────────────────────────────
        # 5. SCALE FEATURES (OBLIGATOIRE POUR LSTM)
        # ────────────────────────────────────────────────────────────────────
        scaler = StandardScaler()

        # Fit sur train UNIQUEMENT (NO LOOKAHEAD)
        X_train_scaled = scaler.fit_transform(X_train)

        # Transform test (utilise mean/std du train)
        X_test_scaled = scaler.transform(X.loc[[formation_date]].values)

        # ────────────────────────────────────────────────────────────────────
        # 5.1 TRAIN LSTM
        # ────────────────────────────────────────────────────────────────────
        model = build_lstm_model(n_features=X_train_scaled.shape[1])

        # Reshape for LSTM (samples, timesteps, features) - APRÈS scaling
        X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
        X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

        
        early_stop = EarlyStopping(
            monitor='loss',
            patience=3,
            restore_best_weights=True,
            verbose=0
        )
        
        model.fit(
            X_train_lstm, y_train,
            epochs=30,
            batch_size=32,
            verbose=0,
            callbacks=[early_stop]
        )
        
        # ────────────────────────────────────────────────────────────────────
        # 6. PREDICT
        # ────────────────────────────────────────────────────────────────────
        y_proba = float(model.predict(X_test_lstm, verbose=0)[0, 0])
        y_pred = int(y_proba > 0.5)
        
        # ────────────────────────────────────────────────────────────────────
        # 7. RETURN RESULTS
        # ────────────────────────────────────────────────────────────────────
        result = {
            'formation_date': formation_date,
            'year': test_year,
            'actual_regime': int(y_test_val),
            'predicted_regime': y_pred,
            'proba_high_vol': y_proba,
            'strategy': strategy,
            'train_start': train_start,
            'train_end': train_end,
            'train_samples': len(y_train),
            'correct': int(y_test_val == y_pred)
        }
        
        # Cleanup
        del model
        tf.keras.backend.clear_session()
        
        return result
        
    except Exception as e:
        return None


# ════════════════════════════════════════════════════════════════════════════
# EXECUTION
# ════════════════════════════════════════════════════════════════════════════

if USE_MODAL:
    print("\n⚠️  USE_MODAL=True detected, skipping local execution")

else:
    from joblib import Parallel, delayed, cpu_count
    import tensorflow as tf
    import time
    
    tf.get_logger().setLevel('ERROR')
    
    # ────────────────────────────────────────────────────────────────────────
    # CONFIGURATION
    # ────────────────────────────────────────────────────────────────────────
    N_CORES = min(8, cpu_count())
    
    print(f"\n{'='*100}")
    print(f"🚀 LSTM TRAINING : {STRATEGY.upper()}")
    print(f"{'='*100}")
    print(f"   • Strategy        : {STRATEGY}")
    print(f"   • Formations      : {FIRST_TEST_YEAR} → {LAST_TEST_YEAR} ({LAST_TEST_YEAR - FIRST_TEST_YEAR + 1} years)")
    print(f"   • CPU cores       : {N_CORES}")
    print(f"   • Expected time   : ~20-30 seconds")
    print(f"{'='*100}\n")
    
    # ────────────────────────────────────────────────────────────────────────
    # SERIALIZE DATA
    # ────────────────────────────────────────────────────────────────────────
    X_values = X.values
    X_index = X.index.astype(str).tolist()
    y_values = y.values
    y_index = y.index.astype(str).tolist()
    
    # ────────────────────────────────────────────────────────────────────────
    # PREPARE ARGUMENTS
    # ────────────────────────────────────────────────────────────────────────
    test_years = range(FIRST_TEST_YEAR, LAST_TEST_YEAR + 1)
    
    args_list = [
        (year, STRATEGY, X_values, X_index, y_values, y_index)
        for year in test_years
    ]
    
    # ────────────────────────────────────────────────────────────────────────
    # PARALLEL TRAINING
    # ────────────────────────────────────────────────────────────────────────
    start_time = time.time()
    
    print(f"🔥 Training {len(test_years)} formations on {N_CORES} cores...\n")
    
    results = Parallel(n_jobs=N_CORES, verbose=10)(
        delayed(train_single_formation_annual)(args) 
        for args in args_list
    )
    
    # ────────────────────────────────────────────────────────────────────────
    # COLLECT RESULTS
    # ────────────────────────────────────────────────────────────────────────
    predictions = [r for r in results if r is not None]
    df_predictions = pd.DataFrame(predictions)
    
    elapsed = time.time() - start_time
    
    # ────────────────────────────────────────────────────────────────────────
    # SUMMARY
    # ────────────────────────────────────────────────────────────────────────
    print(f"\n{'='*100}")
    print(f"✅ TRAINING COMPLETE : {STRATEGY.upper()}")
    print(f"{'='*100}")
    
    if len(df_predictions) > 0:
        accuracy = df_predictions['correct'].mean()
        
        print(f"\n📊 RESULTS :")
        print(f"   • Valid predictions : {len(df_predictions)}/{len(test_years)}")
        print(f"   • Accuracy          : {accuracy:.2%}")
        print(f"   • Period            : {df_predictions['formation_date'].min().date()} → {df_predictions['formation_date'].max().date()}")
        print(f"   • Training time     : {elapsed:.1f}s")
        print(f"   • Avg per formation : {elapsed/len(df_predictions):.2f}s")
        
        print(f"\n📊 TRAIN SAMPLES STATISTICS :")
        print(f"   • Min    : {df_predictions['train_samples'].min()}")
        print(f"   • Max    : {df_predictions['train_samples'].max()}")
        print(f"   • Mean   : {df_predictions['train_samples'].mean():.1f}")
        
        if STRATEGY == 'expanding':
            is_growing = df_predictions['train_samples'].is_monotonic_increasing
            print(f"   • Growing: {is_growing} {'✅' if is_growing else '❌'}")
        elif STRATEGY == 'rolling_10y':
            is_constant = df_predictions['train_samples'].std() < 5
            print(f"   • Constant (~120): {is_constant} {'✅' if is_constant else '❌'}")
    else:
        print(f"\n⚠️  No valid predictions generated")
    
    print(f"\n{'='*100}\n")



🚀 LSTM TRAINING : ROLLING_10Y
   • Strategy        : rolling_10y
   • Formations      : 1980 → 2023 (44 years)
   • CPU cores       : 8
   • Expected time   : ~20-30 seconds

🔥 Training 44 formations on 8 cores...



[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:    6.9s
[Parallel(n_jobs=8)]: Done   9 tasks      | elapsed:   14.4s
[Parallel(n_jobs=8)]: Done  16 tasks      | elapsed:   14.9s
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:   26.4s
[Parallel(n_jobs=8)]: Done  34 out of  44 | elapsed:   33.4s remaining:    9.8s
[Parallel(n_jobs=8)]: Done  39 out of  44 | elapsed:   34.0s remaining:    4.4s



✅ TRAINING COMPLETE : ROLLING_10Y

📊 RESULTS :
   • Valid predictions : 44/44
   • Accuracy          : 72.73%
   • Period            : 1980-06-30 → 2023-06-30
   • Training time     : 37.5s
   • Avg per formation : 0.85s

📊 TRAIN SAMPLES STATISTICS :
   • Min    : 120
   • Max    : 120
   • Mean   : 120.0
   • Constant (~120): True ✅




[Parallel(n_jobs=8)]: Done  44 out of  44 | elapsed:   37.5s finished


In [19]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 7 : MODEL COMPARISON (LSTM vs XGBoost vs RF vs Logistic)
# ════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*100}")
print(f"📊 FINAL MODEL COMPARISON : {STRATEGY.upper()} vs TREE MODELS")
print(f"{'='*100}\n")

# ════════════════════════════════════════════════════════════════════════════
# LOAD METRICS FROM OTHER MODELS
# ════════════════════════════════════════════════════════════════════════════

import json

with open('data/ml_data/models/logistic/metrics.json') as f:
    logistic_metrics = json.load(f)['performance']['overall']

with open('data/ml_data/models/random_forest/metrics.json') as f:
    rf_metrics = json.load(f)['performance']['overall']

with open('data/ml_data/models/xgboost/metrics.json') as f:
    xgb_metrics = json.load(f)['performance']['overall']

# ════════════════════════════════════════════════════════════════════════════
# GET CURRENT LSTM METRICS
# ════════════════════════════════════════════════════════════════════════════

if len(df_predictions) > 0:
    y_actual = df_predictions['actual_regime'].values
    y_pred = df_predictions['predicted_regime'].values
    y_proba = df_predictions['proba_high_vol'].values
    
    lstm_metrics = {
        'accuracy': float(accuracy_score(y_actual, y_pred)),
        'precision': float(precision_score(y_actual, y_pred, zero_division=0)),
        'recall': float(recall_score(y_actual, y_pred, zero_division=0)),
        'f1': float(f1_score(y_actual, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_actual, y_proba))
    }
    
    # ════════════════════════════════════════════════════════════════════════
    # BUILD COMPARISON TABLE
    # ════════════════════════════════════════════════════════════════════════
    
    final_comparison = pd.DataFrame({
        'Logistic L2': logistic_metrics,
        'Random Forest': rf_metrics,
        'XGBoost': xgb_metrics,
        f'LSTM ({STRATEGY})': lstm_metrics
    }).T
    
    print(final_comparison.to_string())
    
    # ════════════════════════════════════════════════════════════════════════
    # IDENTIFY WINNERS
    # ════════════════════════════════════════════════════════════════════════
    
    print(f"\n🏆 OVERALL WINNER :")
    for metric in ['accuracy', 'roc_auc']:
        winner = final_comparison[metric].idxmax()
        print(f"   • {metric.upper():<12} : {winner:<25} ({final_comparison.loc[winner, metric]:.4f})")
    
    # ════════════════════════════════════════════════════════════════════════
    # STRENGTHS SUMMARY
    # ════════════════════════════════════════════════════════════════════════
    
    print(f"\n📊 STRENGTHS BY MODEL :")
    print(f"   • Logistic L2          : Speed, interpretability, linear patterns")
    print(f"   • Random Forest        : Non-linearity, robustness, feature interactions")
    print(f"   • XGBoost              : State-of-the-art trees, regularization")
    print(f"   • LSTM ({STRATEGY:>10}) : Temporal sequences, regime transitions")
    
    # ════════════════════════════════════════════════════════════════════════
    # LSTM SPECIFIC INSIGHTS
    # ════════════════════════════════════════════════════════════════════════
    
    lstm_rank_acc = (final_comparison['accuracy'] > lstm_metrics['accuracy']).sum() + 1
    lstm_rank_auc = (final_comparison['roc_auc'] > lstm_metrics['roc_auc']).sum() + 1
    
    print(f"\n📊 LSTM {STRATEGY.upper()} PERFORMANCE :")
    print(f"   • Rank (Accuracy)  : {lstm_rank_acc}/4")
    print(f"   • Rank (ROC-AUC)   : {lstm_rank_auc}/4")
    
    if lstm_rank_acc <= 2 or lstm_rank_auc <= 2:
        print(f"   • Status           : ✅ COMPETITIVE (Top 2)")
    elif lstm_rank_acc == 3 or lstm_rank_auc == 3:
        print(f"   • Status           : 🟡 MODERATE (3rd place)")
    else:
        print(f"   • Status           : ❌ UNDERPERFORMING (4th place)")
    
    print("\n" + "="*100)

else:
    print(f"\n⚠️  No predictions available for comparison")
    print("="*100)



📊 FINAL MODEL COMPARISON : ROLLING_10Y vs TREE MODELS

                    accuracy  precision  recall        f1   roc_auc
Logistic L2         0.727273   0.666667  0.5000  0.571429  0.839286
Random Forest       0.704545   0.666667  0.3750  0.480000  0.716518
XGBoost             0.659091   0.538462  0.4375  0.482759  0.732143
LSTM (rolling_10y)  0.727273   0.625000  0.6250  0.625000  0.707589

🏆 OVERALL WINNER :
   • ACCURACY     : Logistic L2               (0.7273)
   • ROC_AUC      : Logistic L2               (0.8393)

📊 STRENGTHS BY MODEL :
   • Logistic L2          : Speed, interpretability, linear patterns
   • Random Forest        : Non-linearity, robustness, feature interactions
   • XGBoost              : State-of-the-art trees, regularization
   • LSTM (rolling_10y) : Temporal sequences, regime transitions

📊 LSTM ROLLING_10Y PERFORMANCE :
   • Rank (Accuracy)  : 1/4
   • Rank (ROC-AUC)   : 4/4
   • Status           : ✅ COMPETITIVE (Top 2)



In [20]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 6 : EXPORT RESULTS (STRATEGY-SPECIFIC)
# ════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*100}")
print(f"💾 EXPORTING RESULTS : {STRATEGY.upper()}")
print(f"{'='*100}\n")

from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix
)

# ════════════════════════════════════════════════════════════════════════════
# 1. CREATE DIRECTORY STRUCTURE
# ════════════════════════════════════════════════════════════════════════════

BASE_DIR = Path('data/ml_data/models/lstm')
STRATEGY_DIR = BASE_DIR / STRATEGY

STRATEGY_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Directory created:")
print(f"   {STRATEGY_DIR}/\n")

# ════════════════════════════════════════════════════════════════════════════
# 2. EXPORT PREDICTIONS
# ════════════════════════════════════════════════════════════════════════════

if len(df_predictions) > 0:
    pred_path = STRATEGY_DIR / 'predictions.parquet'
    df_predictions.to_parquet(pred_path, index=False)
    
    print(f"📊 Predictions exported:")
    print(f"   • File        : {pred_path}")
    print(f"   • Rows        : {len(df_predictions)}")
    print(f"   • Columns     : {list(df_predictions.columns)}\n")

    # ════════════════════════════════════════════════════════════════════════
    # 3. COMPUTE METRICS
    # ════════════════════════════════════════════════════════════════════════
    
    y_actual = df_predictions['actual_regime'].values
    y_pred = df_predictions['predicted_regime'].values
    y_proba = df_predictions['proba_high_vol'].values
    
    # ════════════════════════════════════════════════════════════════════════════
    # 4. BUILD METRICS JSON (WITH EXTENDED METRICS)
    # ════════════════════════════════════════════════════════════════════════════
    
    # Standard metrics
    metrics = {
        'accuracy': float(accuracy_score(y_actual, y_pred)),
        'precision': float(precision_score(y_actual, y_pred, zero_division=0)),
        'recall': float(recall_score(y_actual, y_pred, zero_division=0)),
        'f1': float(f1_score(y_actual, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_actual, y_proba))
    }
    
    # ════════════════════════════════════════════════════════════════════════════
    # EXTENDED METRICS FOR PUBLICATION
    # ════════════════════════════════════════════════════════════════════════════
    
    from sklearn.metrics import balanced_accuracy_score, brier_score_loss, log_loss
    
    extended_metrics = {
        'balanced_accuracy': float(balanced_accuracy_score(y_actual, y_pred)),
        'auc': metrics['roc_auc'],
        'f1_macro': float(f1_score(y_actual, y_pred, average='macro', zero_division=0)),
        'brier_score': float(brier_score_loss(y_actual, y_proba)),
        'log_loss': float(log_loss(y_actual, np.column_stack([1-y_proba, y_proba]))),
        'n_obs': len(y_actual)
    }
    
    print(f"\n📊 EXTENDED METRICS (FOR PUBLICATION) :")
    for metric, value in extended_metrics.items():
        print(f"   • {metric:<20} : {value:.4f}")
    
    # ════════════════════════════════════════════════════════════════════════════
    # BUILD FULL METRICS DICT
    # ════════════════════════════════════════════════════════════════════════════
    
    metrics_dict = {
        'model': 'LSTM',
        'strategy': STRATEGY,
        'architecture': {
            'layers': 'LSTM(32) + Dropout(0.2) + Dense(1, sigmoid)',
            'optimizer': 'Adam(lr=0.001)',
            'loss': 'binary_crossentropy',
            'epochs': 30,
            'batch_size': 32,
            'early_stopping': 'patience=3'
        },
        'validation': {
            'type': 'walk_forward',
            'rebalancing': 'annual',
            'first_test': f'{FIRST_TEST_YEAR}-06-30',
            'last_test': f'{LAST_TEST_YEAR}-06-30',
            'n_formations': len(df_predictions)
        },
        'train_window': {
            'type': 'expanding' if STRATEGY == 'expanding' else 'rolling',
            'size': 'variable' if STRATEGY == 'expanding' else '120 months',
            'min_samples': int(df_predictions['train_samples'].min()),
            'max_samples': int(df_predictions['train_samples'].max()),
            'mean_samples': float(df_predictions['train_samples'].mean())
        },
        'performance': {
            'overall': metrics,
            'extended': extended_metrics,  # ✅ AJOUT ICI
            'by_regime': {
                'regime_0': {
                    'count': int((y_actual == 0).sum()),
                    'accuracy': float(df_predictions[df_predictions['actual_regime'] == 0]['correct'].mean())
                },
                'regime_1': {
                    'count': int((y_actual == 1).sum()),
                    'accuracy': float(df_predictions[df_predictions['actual_regime'] == 1]['correct'].mean())
                }
            }
        },
        'temporal_stability': {
            'early_period': {
                'years': '≤1990',
                'accuracy': float(df_predictions[df_predictions['year'] <= 1990]['correct'].mean())
            },
            'late_period': {
                'years': '≥2000',
                'accuracy': float(df_predictions[df_predictions['year'] >= 2000]['correct'].mean())
            }
        }
    }
    
    # ════════════════════════════════════════════════════════════════════════
    # 5. EXPORT METRICS
    # ════════════════════════════════════════════════════════════════════════
    
    metrics_path = STRATEGY_DIR / 'metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics_dict, f, indent=2)
    
    print(f"📈 Metrics exported:")
    print(f"   • File        : {metrics_path}")
    print(f"   • Accuracy    : {metrics['accuracy']:.4f}")
    print(f"   • ROC-AUC     : {metrics['roc_auc']:.4f}\n")
    
    # ════════════════════════════════════════════════════════════════════════
    # 6. DISPLAY SUMMARY
    # ════════════════════════════════════════════════════════════════════════
    
    print(f"{'='*100}")
    print(f"✅ EXPORT COMPLETE : {STRATEGY.upper()}")
    print(f"{'='*100}")
    
    print(f"\n📁 Output structure:")
    print(f"   {STRATEGY_DIR}/")
    print(f"   ├── predictions.parquet  ({len(df_predictions)} rows)")
    print(f"   └── metrics.json")
    
    print(f"\n📊 Performance summary:")
    print(f"   • Accuracy    : {metrics['accuracy']:.2%}")
    print(f"   • Precision   : {metrics['precision']:.2%}")
    print(f"   • Recall      : {metrics['recall']:.2%}")
    print(f"   • F1-Score    : {metrics['f1']:.2%}")
    print(f"   • ROC-AUC     : {metrics['roc_auc']:.4f}")
    
    print(f"\n🎯 Next steps:")
    if STRATEGY == 'expanding':
        print(f"   1. Change STRATEGY = 'rolling_10y' in CELL 4")
        print(f"   2. Re-run CELL 4, CELL 5, CELL 6")
        print(f"   3. Compare expanding vs rolling_10y performance")
    else:
        print(f"   1. Load predictions:")
        print(f"      pd.read_parquet('{STRATEGY_DIR}/predictions.parquet')")
        print(f"   2. Compare with other models (08_1/08_2/08_3)")
        print(f"   3. Build ensemble strategy (Notebook 09)")
    
    print(f"\n{'='*100}\n")

else:
    print(f"⚠️  No predictions to export for {STRATEGY}")
    print(f"{'='*100}\n")



💾 EXPORTING RESULTS : ROLLING_10Y

✅ Directory created:
   data/ml_data/models/lstm/rolling_10y/

📊 Predictions exported:
   • File        : data/ml_data/models/lstm/rolling_10y/predictions.parquet
   • Rows        : 44
   • Columns     : ['formation_date', 'year', 'actual_regime', 'predicted_regime', 'proba_high_vol', 'strategy', 'train_start', 'train_end', 'train_samples', 'correct']


📊 EXTENDED METRICS (FOR PUBLICATION) :
   • balanced_accuracy    : 0.7054
   • auc                  : 0.7076
   • f1_macro             : 0.7054
   • brier_score          : 0.2116
   • log_loss             : 0.6492
   • n_obs                : 44.0000
📈 Metrics exported:
   • File        : data/ml_data/models/lstm/rolling_10y/metrics.json
   • Accuracy    : 0.7273
   • ROC-AUC     : 0.7076

✅ EXPORT COMPLETE : ROLLING_10Y

📁 Output structure:
   data/ml_data/models/lstm/rolling_10y/
   ├── predictions.parquet  (44 rows)
   └── metrics.json

📊 Performance summary:
   • Accuracy    : 72.73%
   • Precision